# Priv-ViT Training Notebook

End-to-end training and evaluation for privacy-preserving egocentric activity recognition on EPIC-KITCHENS-100.

**What you will run through (in order):**
1. Mount Drive / load paths — point at your `cv_project/` data folder
2. Validate dataset — confirm `.npz` clips and annotation CSVs are present
3. Privacy transforms — on-the-fly Level 0/1/2 degradation (blur, downscale, JPEG)
4. Build Priv-ViT — factorised ViViT-S student (~30M params)
5. Load SlowFast-R50 — frozen teacher for cross-fidelity distillation
6. Train — KL + CE loss, cosine schedule, mixed precision
7. Evaluate — Top-1/Top-5 at all privacy levels, GFLOPs, FPS benchmarks
8. Plot results — training curves and privacy-level comparison charts

**Prerequisite:** Run `process_dataset_local.py` locally, then upload the `cv_project/` folder to Google Drive (Colab) or set `DRIVE_ROOT` to `./cv_project` for local GPU training.


## Configuration

Edit `DRIVE_ROOT` before running. On Colab, set it to your uploaded folder (e.g. `/content/drive/MyDrive/cv_project`).


In [ ]:
# --- User configuration (edit before running) ---
# Path to cv_project/ (output of process_dataset_local.py)
DRIVE_ROOT = './cv_project'

# Colab only: local disk cache for faster I/O during training
LOCAL_CACHE_DIR = '/content/processed_clips'

# Set True to copy processed clips to LOCAL_CACHE_DIR on Colab
USE_COLAB_LOCAL_CACHE = True


## Cell 1 — Setup & Installation

In [ ]:
import os, sys, subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Data root: {DRIVE_ROOT}")
print(f"Running in Colab: {IN_COLAB}")

def install_packages():
    pkgs = [
        'pytorchvideo',
        'timm',
        'fvcore',
        'opencv-python-headless',
        'insightface',
        'onnxruntime-gpu',
        'pandas',
        'matplotlib',
        'tqdm',
        'tensorboard',
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
    print("All packages installed.")

install_packages()

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import numpy as np
import cv2
import pandas as pd
import json
import glob
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 2 — Configuration

In [ ]:
CONFIG = {
    # === Paths (all on Google Drive for persistence) ===
    'drive_root': DRIVE_ROOT,
    'annotations_dir': os.path.join(DRIVE_ROOT, 'annotations'),
    'raw_frames_dir': os.path.join(DRIVE_ROOT, 'raw_frames'),
    'processed_dir': os.path.join(DRIVE_ROOT, 'processed_clips'),
    'checkpoint_dir': os.path.join(DRIVE_ROOT, 'checkpoints'),
    'log_dir': os.path.join(DRIVE_ROOT, 'logs'),

    # === Dataset ===
    'participants': ['P01', 'P06', 'P07', 'P09', 'P11', 'P12'],  # Pre-downloaded on Drive
    'num_frames': 16,           # Frames per clip
    'num_verb_classes': 97,
    'num_noun_classes': 300,
    'task': 'verb',             # We classify verbs

    # === Privacy ===
    'privacy_level': 2,         # 0, 1, or 2
    # Level 0: 224x224 RGB (baseline)
    # Level 1: 112x112, face regions Gaussian-blurred (sigma=5)
    # Level 2: 56x56, full-frame blur (sigma=15), JPEG quality 10

    # === Model (Priv-ViT / ViViT-S) ===
    'img_size': 224,            # Input resolution before privacy transform
    'tubelet_size': (4, 16, 16),  # (t, h, w) for 3D patch embedding
    'embed_dim': 384,
    'num_heads': 6,
    'depth': 12,
    'mlp_ratio': 4.0,
    'drop_rate': 0.1,
    'attn_drop_rate': 0.0,

    # === Teacher (SlowFast-R50) ===
    'teacher_finetune_epochs': 10,
    'teacher_lr': 1e-4,

    # === Distillation ===
    'alpha': 0.5,               # Weight for KL loss vs CE loss
    'temperature': 4.0,         # Softmax temperature for distillation

    # === Training ===
    'epochs': 30,
    'batch_size': 8,
    'lr': 1e-4,
    'weight_decay': 0.05,
    'warmup_epochs': 3,
    'use_amp': True,            # Mixed precision

    # === Eval ===
    'eval_batch_size': 16,
}

# Create directories
for key in ['annotations_dir', 'raw_frames_dir', 'processed_dir',
            'checkpoint_dir', 'log_dir']:
    os.makedirs(CONFIG[key], exist_ok=True)

print("Configuration set.")
print(f"Privacy Level: {CONFIG['privacy_level']}")
print(f"Participants: {CONFIG['participants']}")
print(f"Epochs: {CONFIG['epochs']}, Batch size: {CONFIG['batch_size']}")

## Cell 3 — Dataset Validation

Loads EPIC-KITCHENS-100 annotations and validates that pre-processed clips
are present on Drive. Clips must be generated locally using
`process_dataset_local.py` and uploaded to Drive before running this notebook.

In [ ]:
# --- 3a: Download annotations & create custom train/val split ---
# NOTE: The official EPIC-KITCHENS-100 validation set uses EPIC-55 videos
# (P0X_0XX IDs) which we don't have. We only have extension videos (P0X_1XX).
# So we create a custom 80/20 split by video from the available data.
# This matches the split used in process_dataset_local.py (seed=42).

ANNOTATIONS_REPO = 'https://github.com/epic-kitchens/epic-kitchens-100-annotations.git'
ann_dir = CONFIG['annotations_dir']

if not os.path.exists(os.path.join(ann_dir, 'EPIC_100_train.csv')):
    print("📥 Downloading EPIC-KITCHENS-100 annotations...")
    subprocess.run(['git', 'clone', ANNOTATIONS_REPO, ann_dir],
                   check=True, capture_output=True)
    print("Annotations downloaded.")
else:
    print("Annotations already present on Drive.")

# Load both official splits and merge
official_train = pd.read_csv(os.path.join(ann_dir, 'EPIC_100_train.csv'))
official_val = pd.read_csv(os.path.join(ann_dir, 'EPIC_100_validation.csv'))
all_segments = pd.concat([official_train, official_val], ignore_index=True)

# Custom 80/20 split from available extension videos only
VAL_SPLIT_RATIO = 0.2
SPLIT_SEED = 42
np.random.seed(SPLIT_SEED)

available_pids = set(CONFIG['participants'])
available_mask = (
    all_segments['participant_id'].isin(available_pids) &
    all_segments['video_id'].str.contains(r'_1\d{2}$', regex=True)
)
filtered = all_segments[available_mask].copy()

train_vids, val_vids = [], []
for pid in sorted(available_pids):
    pid_videos = sorted(filtered[filtered['participant_id'] == pid]['video_id'].unique())
    n_val = max(1, int(len(pid_videos) * VAL_SPLIT_RATIO))
    shuffled = np.random.permutation(pid_videos)
    val_vids.extend(shuffled[:n_val])
    train_vids.extend(shuffled[n_val:])

val_vid_set = set(val_vids)
train_df = filtered[~filtered['video_id'].isin(val_vid_set)].copy()
val_df = filtered[filtered['video_id'].isin(val_vid_set)].copy()

print(f"   Custom split: {len(train_df)} train / {len(val_df)} val segments")
print(f"   Videos: {len(train_vids)} train / {len(val_vids)} val")
print(f"   Verb classes: {filtered['verb_class'].nunique()}")

# --- 3b: Validate pre-processed clips on Drive ---
processed_dir = CONFIG['processed_dir']
train_clip_dir = os.path.join(processed_dir, 'train')
val_clip_dir = os.path.join(processed_dir, 'val')

n_train = len(glob.glob(os.path.join(train_clip_dir, '*.npz')))
n_val = len(glob.glob(os.path.join(val_clip_dir, '*.npz')))
train_ok = os.path.exists(os.path.join(train_clip_dir, '.processing_complete'))
val_ok = os.path.exists(os.path.join(val_clip_dir, '.processing_complete'))

if train_ok and val_ok and n_train > 0 and n_val > 0:
    print(f"\n Pre-processed clips found on Drive!")
    print(f"   Train: {n_train} clips, Val: {n_val} clips")
else:
    missing = []
    if not train_ok or n_train == 0:
        missing.append(f"train ({n_train} clips, marker={'Present' if train_ok else 'Not Present'})")
    if not val_ok or n_val == 0:
        missing.append(f"val ({n_val} clips, marker={'Present' if val_ok else 'Not Present'})")
    print(f"\n Pre-processed clips missing: {', '.join(missing)}")
    print(f"   Please run process_dataset_local.py locally, then upload")
    print(f"   the cv_project/ folder to Google Drive at: My Drive/cv_project/")
    raise FileNotFoundError(
        "Pre-processed clips not found on Drive. "
        "Run process_dataset_local.py locally and upload to Drive."
    )

## Cell 4 — Privacy Transforms

Three privacy levels applied on-the-fly:
- **Level 0**: 224×224 RGB (no modification)
- **Level 1**: 112×112, face regions Gaussian-blurred (σ=5) via RetinaFace
- **Level 2**: 56×56, full-frame blur (σ=15), JPEG quality 10

In [ ]:
class FaceDetector:
    """Lightweight face detector using InsightFace's RetinaFace."""

    def __init__(self):
        self._app = None

    def _init_model(self):
        if self._app is None:
            try:
                from insightface.app import FaceAnalysis
                self._app = FaceAnalysis(
                    allowed_modules=['detection'],
                    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
                )
                self._app.prepare(ctx_id=0, det_size=(160, 160))
                print("RetinaFace face detector loaded.")
            except Exception as e:
                print(f"  Could not load RetinaFace: {e}")
                print("   Falling back to OpenCV DNN face detector.")
                self._app = 'opencv_fallback'

    def detect_faces(self, img_rgb):
        """Returns list of (x1, y1, x2, y2) bounding boxes."""
        self._init_model()

        if self._app == 'opencv_fallback':
            return self._detect_opencv(img_rgb)

        try:
            faces = self._app.get(img_rgb)
            return [face.bbox.astype(int).tolist() for face in faces]
        except Exception:
            return []

    def _detect_opencv(self, img_rgb):
        """Fallback using OpenCV's DNN face detector."""
        h, w = img_rgb.shape[:2]
        blob = cv2.dnn.blobFromImage(img_rgb, 1.0, (300, 300),
                                     (104.0, 177.0, 123.0))
        if not hasattr(self, '_net'):
            prototxt = cv2.data.haarcascades  # Fallback to Haar
            # Simple Haar cascade as last resort
            self._cascade = cv2.CascadeClassifier(
                cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
            )
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        rects = self._cascade.detectMultiScale(gray, 1.1, 4, minSize=(20, 20))
        return [[x, y, x + w_, y + h_] for (x, y, w_, h_) in rects]


face_detector = FaceDetector()


class PrivacyTransform:
    """Apply privacy-level transforms to a batch of frames.

    Input:  numpy array (T, H, W, C) uint8 RGB
    Output: torch tensor (C, T, H', W') float32 normalized
    """

    LEVEL_CONFIGS = {
        0: {'size': 224, 'face_blur': False, 'frame_blur': 0,  'jpeg_quality': None},
        1: {'size': 112, 'face_blur': True,  'frame_blur': 0,  'jpeg_quality': None},
        2: {'size': 56,  'face_blur': False, 'frame_blur': 15, 'jpeg_quality': 10},
    }

    def __init__(self, level=0):
        assert level in (0, 1, 2), f"Privacy level must be 0, 1, or 2, got {level}"
        self.level = level
        self.cfg = self.LEVEL_CONFIGS[level]
        self.normalize = T.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __call__(self, frames_np):
        """
        Args:
            frames_np: (T, H, W, C) uint8 numpy array
        Returns:
            (C, T, H', W') float32 torch tensor, normalized
        """
        T_len = frames_np.shape[0]
        processed = []

        for t in range(T_len):
            frame = frames_np[t].copy()  # (H, W, C)

            # Face blur (Level 1 only)
            if self.cfg['face_blur']:
                boxes = face_detector.detect_faces(frame)
                for (x1, y1, x2, y2) in boxes:
                    x1, y1 = max(0, x1), max(0, y1)
                    x2 = min(frame.shape[1], x2)
                    y2 = min(frame.shape[0], y2)
                    face_region = frame[y1:y2, x1:x2]
                    if face_region.size > 0:
                        ksize = max(3, int(5 * 6) | 1)  # sigma=5
                        frame[y1:y2, x1:x2] = cv2.GaussianBlur(
                            face_region, (ksize, ksize), 5
                        )

            # Full-frame Gaussian blur (Level 2)
            if self.cfg['frame_blur'] > 0:
                sigma = self.cfg['frame_blur']
                ksize = max(3, int(sigma * 6) | 1)
                frame = cv2.GaussianBlur(frame, (ksize, ksize), sigma)

            # JPEG compression (Level 2)
            if self.cfg['jpeg_quality'] is not None:
                encode_param = [int(cv2.IMWRITE_JPEG_QUALITY),
                                self.cfg['jpeg_quality']]
                _, enc = cv2.imencode('.jpg', frame, encode_param)
                frame = cv2.imdecode(enc, cv2.IMREAD_COLOR)

            # Resize to target resolution
            frame = cv2.resize(frame, (self.cfg['size'], self.cfg['size']),
                               interpolation=cv2.INTER_LINEAR)

            processed.append(frame)

        # Stack → (T, H, W, C) → (C, T, H, W) float32
        frames_arr = np.stack(processed, axis=0)
        tensor = torch.from_numpy(frames_arr).float().permute(3, 0, 1, 2) / 255.0

        # Normalize each frame
        C, T_dim, H, W = tensor.shape
        tensor = tensor.view(C, T_dim, H * W)
        for c in range(C):
            tensor[c] = (tensor[c] - self.normalize.mean[c]) / self.normalize.std[c]
        tensor = tensor.view(C, T_dim, H, W)

        return tensor

    def __repr__(self):
        return f"PrivacyTransform(level={self.level}, size={self.cfg['size']})"


# Quick visual test
print("Privacy transform configs:")
for lvl in range(3):
    pt = PrivacyTransform(level=lvl)
    print(f"  Level {lvl}: {pt.cfg}")

## Cell 5 — Dataset & DataLoader

In [ ]:
class EpicKitchensClipDataset(Dataset):
    """Loads pre-processed 16-frame .npz clips from Drive."""

    def __init__(self, split='train', privacy_level=0, augment=True):
        split_dir = os.path.join(CONFIG['processed_dir'], split)
        self.clip_paths = sorted(glob.glob(os.path.join(split_dir, '*.npz')))
        self.privacy_transform = PrivacyTransform(level=privacy_level)
        self.augment = augment and (split == 'train')

        if len(self.clip_paths) == 0:
            print(f"No clips found in {split_dir}. "
                  f"Run Cell 3 to process the dataset first.")

    def __len__(self):
        return len(self.clip_paths)

    def __getitem__(self, idx):
        data = np.load(self.clip_paths[idx])
        frames = data['frames']       # (T, H, W, C) uint8
        verb_class = int(data['verb_class'])

        # Training augmentations (applied before privacy transform)
        if self.augment:
            # Random horizontal flip
            if np.random.random() > 0.5:
                frames = frames[:, :, ::-1, :].copy()

            # Random temporal jitter (shift by ±1 frame)
            if np.random.random() > 0.5 and frames.shape[0] > 2:
                shift = np.random.randint(-1, 2)
                frames = np.roll(frames, shift, axis=0)

            # Random color jitter (brightness, contrast)
            if np.random.random() > 0.5:
                factor = np.random.uniform(0.8, 1.2)
                frames = np.clip(frames * factor, 0, 255).astype(np.uint8)

        # Apply privacy transform → (C, T, H', W') tensor
        clip_tensor = self.privacy_transform(frames)

        return clip_tensor, verb_class

    def __repr__(self):
        return (f"EpicKitchensClipDataset(clips={len(self)}, "
                f"privacy={self.privacy_transform.level})")


def build_dataloaders(privacy_level):
    """Create train and val dataloaders for a given privacy level."""
    train_ds = EpicKitchensClipDataset('train', privacy_level, augment=True)
    val_ds = EpicKitchensClipDataset('val', privacy_level, augment=False)

    # num_workers=2 to avoid Drive I/O hangs on Colab
    # persistent_workers disabled — can cause freezes with Drive FUSE
    train_loader = DataLoader(
        train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
        prefetch_factor=2
    )
    val_loader = DataLoader(
        val_ds, batch_size=CONFIG['eval_batch_size'], shuffle=False,
        num_workers=2, pin_memory=True,
        prefetch_factor=2
    )

    print(f"DataLoaders built (privacy level {privacy_level}):")
    print(f"   Train: {len(train_ds)} clips, {len(train_loader)} batches")
    print(f"   Val:   {len(val_ds)} clips, {len(val_loader)} batches")
    return train_loader, val_loader


# Build for current privacy level
train_loader, val_loader = build_dataloaders(CONFIG['privacy_level'])

## Cell 6 — Priv-ViT Model Architecture

Adapted ViViT-S with:
- **Tubelet Embedding**: `Conv3d(3, 384, kernel=(4,16,16), stride=(4,16,16))`
- **Factorized Self-Attention**: Spatial → Temporal in each block (~40% fewer FLOPs)
- **~30M parameters**, initialized from ImageNet-21K ViT weights

In [ ]:
class TubeletEmbedding(nn.Module):
    """3D patch embedding via Conv3d (tubelet embedding).

    Maps (B, C, T, H, W) → (B, N, D) where N = num_temporal * num_spatial tokens.
    """

    def __init__(self, in_channels=3, embed_dim=384,
                 tubelet_size=(4, 16, 16)):
        super().__init__()
        self.tubelet_size = tubelet_size
        self.proj = nn.Conv3d(
            in_channels, embed_dim,
            kernel_size=tubelet_size,
            stride=tubelet_size
        )
        self.embed_dim = embed_dim

    def forward(self, x):
        # x: (B, C, T, H, W)
        x = self.proj(x)  # (B, D, T', H', W')
        B, D, T, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, T'*H'*W', D)
        return x, (T, H, W)


class SpatialAttention(nn.Module):
    """Multi-head self-attention within each temporal position."""

    def __init__(self, dim, num_heads=6, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, num_temporal, num_spatial):
        """
        x: (B, T*S, D) where T=num_temporal, S=num_spatial
        Attention is computed independently within each temporal slice.
        """
        B, N, D = x.shape
        S = num_spatial
        T = num_temporal

        # Reshape to (B*T, S, D) — treat each temporal slice independently
        x = x.view(B, T, S, D).reshape(B * T, S, D)

        qkv = self.qkv(x).reshape(B * T, S, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B*T, heads, S, head_dim)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B * T, S, D)
        x = self.proj(x)
        x = self.proj_drop(x)

        # Reshape back to (B, T*S, D)
        x = x.view(B, T, S, D).reshape(B, N, D)
        return x


class TemporalAttention(nn.Module):
    """Multi-head self-attention across temporal positions for each spatial location."""

    def __init__(self, dim, num_heads=6, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, num_temporal, num_spatial):
        """
        x: (B, T*S, D)
        Attention across T for each spatial position.
        """
        B, N, D = x.shape
        T = num_temporal
        S = num_spatial

        # Reshape to (B*S, T, D) — treat each spatial position independently
        x = x.view(B, T, S, D).permute(0, 2, 1, 3).reshape(B * S, T, D)

        qkv = self.qkv(x).reshape(B * S, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B*S, heads, T, head_dim)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B * S, T, D)
        x = self.proj(x)
        x = self.proj_drop(x)

        # Reshape back to (B, T*S, D)
        x = x.view(B, S, T, D).permute(0, 2, 1, 3).reshape(B, N, D)
        return x


class MLP(nn.Module):
    """Feed-forward network with GELU activation."""

    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.fc2(x))
        return x


class FactorizedTransformerBlock(nn.Module):
    """One block of the factorized ViViT encoder.

    Applies: Spatial MSA → Temporal MSA → MLP
    Each with pre-norm (LayerNorm) and residual connections.
    """

    def __init__(self, dim, num_heads, mlp_ratio=4.0,
                 drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm_s = nn.LayerNorm(dim)
        self.spatial_attn = SpatialAttention(dim, num_heads, attn_drop, drop)

        self.norm_t = nn.LayerNorm(dim)
        self.temporal_attn = TemporalAttention(dim, num_heads, attn_drop, drop)

        self.norm_mlp = nn.LayerNorm(dim)
        self.mlp = MLP(dim, mlp_ratio, drop)

    def forward(self, x, num_temporal, num_spatial):
        # Spatial self-attention (within each frame)
        x = x + self.spatial_attn(self.norm_s(x), num_temporal, num_spatial)
        # Temporal self-attention (across frames)
        x = x + self.temporal_attn(self.norm_t(x), num_temporal, num_spatial)
        # Feed-forward
        x = x + self.mlp(self.norm_mlp(x))
        return x


class PrivViT(nn.Module):
    """Priv-ViT: Privacy-preserving Video Vision Transformer (ViViT-S variant).

    Architecture:
        - Tubelet embedding (Conv3d 4×16×16)
        - Learnable positional embedding
        - 12 factorized transformer blocks (spatial → temporal attention)
        - Global average pooling → classification head
    """

    def __init__(self, num_classes=97, img_size=224, num_frames=16,
                 tubelet_size=(4, 16, 16), embed_dim=384, depth=12,
                 num_heads=6, mlp_ratio=4.0, drop_rate=0.1, attn_drop=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_frames = num_frames

        # Tubelet embedding
        self.patch_embed = TubeletEmbedding(3, embed_dim, tubelet_size)

        # Calculate number of tokens
        t_tokens = num_frames // tubelet_size[0]
        h_tokens = img_size // tubelet_size[1]
        w_tokens = img_size // tubelet_size[2]
        self.num_temporal = t_tokens
        self.num_spatial = h_tokens * w_tokens
        num_tokens = t_tokens * h_tokens * w_tokens

        # CLS token + positional embedding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_tokens + 1, embed_dim)
        )
        self.pos_drop = nn.Dropout(drop_rate)

        # Transformer blocks with factorized attention
        self.blocks = nn.ModuleList([
            FactorizedTransformerBlock(
                embed_dim, num_heads, mlp_ratio, drop_rate, attn_drop
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize weights
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv3d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    def load_pretrained_2d(self, state_dict_2d):
        """Load ImageNet-pretrained ViT-S/16 weights, inflating 2D→3D."""
        own_state = self.state_dict()

        for name, param in state_dict_2d.items():
            if name not in own_state:
                continue
            if own_state[name].shape == param.shape:
                own_state[name].copy_(param)
            elif 'patch_embed' in name and 'proj.weight' in name:
                # Inflate 2D Conv weight → 3D: (D, C, H, W) → (D, C, T, H, W)
                t_size = self.patch_embed.tubelet_size[0]
                inflated = param.unsqueeze(2).repeat(1, 1, t_size, 1, 1)
                inflated = inflated / t_size  # Normalize
                own_state[name].copy_(inflated)
            elif 'pos_embed' in name:
                # Interpolate positional embedding if sizes differ
                # Skip CLS token, interpolate spatial, tile temporal
                pass  # Handled separately if needed

        self.load_state_dict(own_state, strict=False)
        print("   Loaded pretrained 2D ViT weights (inflated to 3D).")

    def forward(self, x):
        """
        Args:
            x: (B, C, T, H, W) video tensor
        Returns:
            logits: (B, num_classes)
        """
        B = x.shape[0]

        # Adapt input size if privacy level changes resolution
        # The model works with any input resolution; tokens adjust accordingly
        tokens, (T, H, W) = self.patch_embed(x)  # (B, N, D)
        num_temporal = T
        num_spatial = H * W

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)  # (B, 1+N, D)

        # Add positional embedding (interpolate if needed)
        if tokens.shape[1] == self.pos_embed.shape[1]:
            tokens = tokens + self.pos_embed
        else:
            # Interpolate position embedding for different resolutions
            cls_pe = self.pos_embed[:, :1, :]
            spatial_pe = self.pos_embed[:, 1:, :]
            N_new = tokens.shape[1] - 1
            spatial_pe = F.interpolate(
                spatial_pe.transpose(1, 2),
                size=N_new, mode='linear', align_corners=False
            ).transpose(1, 2)
            tokens = tokens + torch.cat([cls_pe, spatial_pe], dim=1)

        tokens = self.pos_drop(tokens)

        # Separate CLS token during factorized attention
        cls_token = tokens[:, :1, :]
        patch_tokens = tokens[:, 1:, :]

        # Transformer blocks
        for block in self.blocks:
            patch_tokens = block(patch_tokens, num_temporal, num_spatial)

        # Re-attach CLS and normalize
        tokens = torch.cat([cls_token, patch_tokens], dim=1)
        tokens = self.norm(tokens)

        # Global average pooling over all tokens (excluding CLS)
        x = tokens[:, 1:, :].mean(dim=1)  # (B, D)

        return self.head(x)


def build_priv_vit():
    """Build Priv-ViT model and optionally load pretrained weights."""
    # Determine input size based on privacy level
    privacy_cfg = PrivacyTransform.LEVEL_CONFIGS[CONFIG['privacy_level']]
    img_size = privacy_cfg['size']

    model = PrivViT(
        num_classes=CONFIG['num_verb_classes'],
        img_size=img_size,
        num_frames=CONFIG['num_frames'],
        tubelet_size=CONFIG['tubelet_size'],
        embed_dim=CONFIG['embed_dim'],
        depth=CONFIG['depth'],
        num_heads=CONFIG['num_heads'],
        mlp_ratio=CONFIG['mlp_ratio'],
        drop_rate=CONFIG['drop_rate'],
        attn_drop=CONFIG['attn_drop_rate'],
    )

    # Try loading pretrained ViT-S/16 weights from timm
    try:
        import timm
        vit_2d = timm.create_model('vit_small_patch16_224', pretrained=True)
        model.load_pretrained_2d(vit_2d.state_dict())
    except Exception as e:
        print(f"  Could not load pretrained weights: {e}")
        print("   Training from scratch.")

    model = model.to(device)
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"\nPriv-ViT built:")
    print(f"   Input: ({CONFIG['num_frames']}×{img_size}×{img_size})")
    print(f"   Params: {total_params:.1f}M total, {trainable:.1f}M trainable")

    return model


student_model = build_priv_vit()

## Cell 7 — Teacher Model (SlowFast-R50)

Frozen SlowFast-R50 pretrained on Kinetics-400.
Fine-tuned briefly on EPIC-KITCHENS verbs, then frozen for distillation.

In [ ]:
class SlowFastTeacher(nn.Module):
    """SlowFast-R50 teacher for knowledge distillation.

    Wraps the pytorchvideo SlowFast model, replacing the classification head
    for EPIC-KITCHENS-100 verb classes.
    """

    def __init__(self, num_classes=97, pretrained=True):
        super().__init__()

        # Load SlowFast-R50 from torch hub
        self.model = torch.hub.load(
            'facebookresearch/pytorchvideo',
            'slowfast_r50',
            pretrained=pretrained
        )

        # Replace classification head
        # The original head outputs 400 classes (Kinetics-400)
        in_features = self.model.blocks[-1].proj.in_features
        self.model.blocks[-1].proj = nn.Linear(in_features, num_classes)

        # SlowFast pathway configuration for 16-frame input
        self.slow_alpha = 4  # Slow pathway temporal stride
        self.slow_frames = CONFIG['num_frames'] // self.slow_alpha  # 4 frames

    def _prepare_slowfast_input(self, x):
        """Convert (B, C, T, H, W) to SlowFast pathway format.

        Returns [slow_pathway, fast_pathway]:
            slow: (B, C, T//alpha, H, W) — subsampled temporal
            fast: (B, C, T, H, W) — full temporal resolution
        """
        B, C, T, H, W = x.shape

        # The pre-trained SLOWFAST_8x8_R50 expects 32 frames in the fast pathway
        # and 8 frames in the slow pathway. We temporally interpolate our 16-frame
        # clips to 32 frames to avoid temporal pooling size mismatches (RuntimeError).
        if T != 32:
            x = F.interpolate(x, size=(32, H, W), mode='trilinear', align_corners=False)
            T = 32

        slow_frames = T // self.slow_alpha  # 32 // 4 = 8 frames

        # Slow pathway: subsample every alpha-th frame
        slow_idx = torch.linspace(0, T - 1, slow_frames).long().to(x.device)
        slow = x[:, :, slow_idx, :, :]

        # Fast pathway: keep all frames
        fast = x

        return [slow, fast]

    def forward(self, x):
        """
        Args:
            x: (B, C, T, H, W) video tensor
        Returns:
            logits: (B, num_classes)
        """
        inputs = self._prepare_slowfast_input(x)
        return self.model(inputs)


def build_teacher():
    """Build and optionally fine-tune the teacher model."""
    teacher_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'teacher_finetuned.pt')

    teacher = SlowFastTeacher(
        num_classes=CONFIG['num_verb_classes'],
        pretrained=True
    ).to(device)

    if os.path.exists(teacher_ckpt):
        teacher.load_state_dict(torch.load(teacher_ckpt, map_location=device))
        print("Teacher loaded from checkpoint (already fine-tuned).")
    else:
        print(" Fine-tuning teacher on EPIC-KITCHENS verbs...")
        finetune_teacher(teacher, teacher_ckpt)

    # Freeze teacher
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad = False

    total_params = sum(p.numel() for p in teacher.parameters()) / 1e6
    print(f" Teacher frozen: {total_params:.1f}M params (no gradients)")
    return teacher


def finetune_teacher(teacher, save_path):
    """Brief fine-tuning of the teacher on EPIC-KITCHENS-100."""
    # Use Level 0 (no privacy) for teacher fine-tuning
    ft_train_loader, ft_val_loader = build_dataloaders(privacy_level=0)

    optimizer = torch.optim.AdamW(
        teacher.parameters(),
        lr=CONFIG['teacher_lr'],
        weight_decay=CONFIG['weight_decay']
    )
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['use_amp'])

    best_acc = 0.0
    for epoch in range(CONFIG['teacher_finetune_epochs']):
        teacher.train()
        total_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(ft_train_loader, desc=f"Teacher Epoch {epoch+1}")
        for clips, labels in pbar:
            clips = clips.to(device)
            labels = labels.to(device)

            with torch.amp.autocast('cuda', enabled=CONFIG['use_amp']):
                logits = teacher(clips)
                loss = criterion(logits, labels)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            _, pred = logits.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=f"{loss.item():.3f}",
                             acc=f"{100.*correct/total:.1f}%")

        # Validation
        val_acc = evaluate_model(teacher, ft_val_loader, topk=(1,))
        print(f"   Teacher Epoch {epoch+1}: "
              f"train_acc={100.*correct/total:.1f}%, val_acc={val_acc:.1f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(teacher.state_dict(), save_path)
            print(f"   Saved best teacher (val_acc={val_acc:.1f}%)")

    # Reload best
    teacher.load_state_dict(torch.load(save_path, map_location=device))
    print(f" Teacher fine-tuned. Best val acc: {best_acc:.1f}%")


@torch.no_grad()
def evaluate_model(model, loader, topk=(1, 5)):
    """Compute Top-1 and Top-5 accuracy."""
    model.eval()
    correct = {k: 0 for k in topk}
    total = 0

    for clips, labels in loader:
        clips = clips.to(device)
        labels = labels.to(device)

        with torch.amp.autocast('cuda', enabled=CONFIG['use_amp']):
            logits = model(clips)

        for k in topk:
            _, pred = logits.topk(k, dim=1)
            correct[k] += pred.eq(labels.unsqueeze(1)).any(dim=1).sum().item()
        total += labels.size(0)

    accs = {k: 100. * correct[k] / max(total, 1) for k in topk}
    return accs[1] if len(topk) == 1 else accs


print("Building teacher model...")
teacher_model = build_teacher()


## Cell 8 — Distillation Loss

Combined loss: `L = α·KL(teacher||student) + (1−α)·CE(y, student)`

In [ ]:
class DistillationLoss(nn.Module):
    """Knowledge distillation loss combining KL divergence and cross-entropy.

    L = alpha * KL_div(softened_teacher || softened_student)
      + (1 - alpha) * CE(ground_truth, student)
    """

    def __init__(self, alpha=0.5, temperature=4.0):
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, labels):
        """
        Args:
            student_logits: (B, C) raw logits from student
            teacher_logits: (B, C) raw logits from frozen teacher
            labels: (B,) ground truth class indices
        """
        T = self.temperature

        # Soft targets from teacher (no gradient)
        soft_teacher = F.softmax(teacher_logits / T, dim=1).detach()
        soft_student = F.log_softmax(student_logits / T, dim=1)

        # KL divergence (scaled by T^2 as per Hinton et al.)
        kl_loss = F.kl_div(soft_student, soft_teacher,
                           reduction='batchmean') * (T ** 2)

        # Hard label cross-entropy
        ce_loss = self.ce_loss(student_logits, labels)

        # Combined loss
        loss = self.alpha * kl_loss + (1 - self.alpha) * ce_loss

        return loss, kl_loss.item(), ce_loss.item()


distill_loss_fn = DistillationLoss(
    alpha=CONFIG['alpha'],
    temperature=CONFIG['temperature']
)
print(f"Distillation loss: α={CONFIG['alpha']}, T={CONFIG['temperature']}")

## Cell 9 — Training Loop

Trains Priv-ViT with knowledge distillation from the frozen teacher.
Features: AMP, cosine LR, checkpoint resume, logging to Drive.

In [ ]:
import math
import time
import shutil
import os
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F

# --- Optional Colab I/O speedup ---
if IN_COLAB and USE_COLAB_LOCAL_CACHE:
    local_processed_dir = LOCAL_CACHE_DIR
else:
    local_processed_dir = CONFIG["processed_dir"]

if IN_COLAB and USE_COLAB_LOCAL_CACHE and local_processed_dir != CONFIG["processed_dir"]:
    if not os.path.exists(local_processed_dir):
        print("Copying processed clips to local Colab disk for faster I/O...")
        shutil.copytree(CONFIG["processed_dir"], local_processed_dir)
        print("Copy complete.")

if CONFIG["processed_dir"] != local_processed_dir:
    CONFIG["processed_dir"] = local_processed_dir
    train_loader, val_loader = build_dataloaders(CONFIG['privacy_level'])

def get_cosine_schedule(optimizer, num_warmup, num_training):
    """Cosine annealing with linear warmup."""
    def lr_lambda(current_step):
        if current_step < num_warmup:
            return float(current_step) / float(max(1, num_warmup))
        progress = float(current_step - num_warmup) / \
                   float(max(1, num_training - num_warmup))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train():
    """Main training loop with distillation."""
    student = student_model
    teacher = teacher_model

    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=CONFIG['lr'],
        weight_decay=CONFIG['weight_decay']
    )

    total_steps = CONFIG['epochs'] * len(train_loader)
    warmup_steps = CONFIG['warmup_epochs'] * len(train_loader)
    scheduler = get_cosine_schedule(optimizer, warmup_steps, total_steps)

    scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['use_amp'])
    writer = SummaryWriter(CONFIG['log_dir'])

    # Resume from checkpoint
    start_epoch = 0
    best_val_acc = 0.0
    ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'priv_vit_latest.pt')

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        student.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = ckpt['epoch'] + 1
        best_val_acc = ckpt.get('best_val_acc', 0.0)
        print(f"Resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.1f}%")

    history = {'train_loss': [], 'train_acc': [], 'val_top1': [], 'val_top5': []}

    print(f"\n Training Priv-ViT for {CONFIG['epochs']} epochs "
          f"(privacy level {CONFIG['privacy_level']})...")
    print(f"   Distillation: α={CONFIG['alpha']}, T={CONFIG['temperature']}")
    print(f"   LR: {CONFIG['lr']}, batch_size: {CONFIG['batch_size']}")
    print("-" * 70)

    for epoch in range(start_epoch, CONFIG['epochs']):
        student.train()
        teacher.eval()

        epoch_loss = 0
        epoch_kl = 0
        epoch_ce = 0
        correct = 0
        total = 0
        t0 = time.time()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")
        for step, (clips, labels) in enumerate(pbar):
            clips = clips.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=CONFIG['use_amp']):
                # Student forward
                student_logits = student(clips)

                # Teacher forward (no grad)
                with torch.no_grad():
                    # Upsample clips to 224x224 for the teacher if needed
                    if clips.shape[-1] != 224 or clips.shape[-2] != 224:
                        teacher_clips = F.interpolate(clips, size=(clips.shape[2], 224, 224), mode='trilinear', align_corners=False)
                    else:
                        teacher_clips = clips
                    teacher_logits = teacher(teacher_clips)

                # Distillation loss
                loss, kl_val, ce_val = distill_loss_fn(
                    student_logits, teacher_logits, labels
                )

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            epoch_loss += loss.item()
            epoch_kl += kl_val
            epoch_ce += ce_val
            _, pred = student_logits.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)

            if step % 20 == 0:
                pbar.set_postfix(
                    loss=f"{loss.item():.3f}",
                    acc=f"{100.*correct/total:.1f}%",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}"
                )

        # Epoch metrics
        train_acc = 100. * correct / total
        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - t0

        # Validation
        val_accs = evaluate_model(student, val_loader, topk=(1, 5))

        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['val_top1'].append(val_accs[1])
        history['val_top5'].append(val_accs[5])

        # Logging
        writer.add_scalar('Loss/train', avg_loss, epoch)
        writer.add_scalar('Loss/kl', epoch_kl / len(train_loader), epoch)
        writer.add_scalar('Loss/ce', epoch_ce / len(train_loader), epoch)
        writer.add_scalar('Acc/train', train_acc, epoch)
        writer.add_scalar('Acc/val_top1', val_accs[1], epoch)
        writer.add_scalar('Acc/val_top5', val_accs[5], epoch)
        writer.add_scalar('LR', scheduler.get_last_lr()[0], epoch)

        print(f"  Epoch {epoch+1}/{CONFIG['epochs']} ({elapsed:.0f}s): "
              f"loss={avg_loss:.3f}, train_acc={train_acc:.1f}%, "
              f"val_top1={val_accs[1]:.1f}%, val_top5={val_accs[5]:.1f}%")

        # Save checkpoint
        ckpt_data = {
            'epoch': epoch,
            'model': student.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'config': CONFIG,
            'history': history,
        }
        torch.save(ckpt_data, ckpt_path)

        if val_accs[1] > best_val_acc:
            best_val_acc = val_accs[1]
            ckpt_data['best_val_acc'] = best_val_acc
            best_path = os.path.join(CONFIG['checkpoint_dir'], 'priv_vit_best.pt')
            torch.save(ckpt_data, best_path)
            print(f"   New best! val_top1={best_val_acc:.1f}%")

    writer.close()
    print(f"\n Training complete! Best val Top-1: {best_val_acc:.1f}%")
    return history


history = train()


## Cell 10 — Evaluation & Metrics

Comprehensive evaluation: Top-1/Top-5 accuracy across all privacy levels,
inference FPS, parameter count, GFLOPs.

In [ ]:
def full_evaluation(model, model_name="Priv-ViT"):
    """Run full evaluation across all privacy levels."""
    print(f"\n{'='*60}")
    print(f"  EVALUATION: {model_name}")
    print(f"{'='*60}")

    results = {}

    for level in [0, 1, 2]:
        print(f"\n--- Privacy Level {level} ---")
        # Use num_workers=0 to avoid multiprocessing assertion errors in Colab loops
        val_ds = EpicKitchensClipDataset('val', privacy_level=level, augment=False)
        vl = DataLoader(
            val_ds, batch_size=CONFIG['eval_batch_size'], shuffle=False,
            num_workers=0, pin_memory=True
        )

        accs = evaluate_model(model, vl, topk=(1, 5))
        results[f'level_{level}_top1'] = accs[1]
        results[f'level_{level}_top5'] = accs[5]
        print(f"  Top-1: {accs[1]:.2f}%  |  Top-5: {accs[5]:.2f}%")

    # Recovery metric
    if results['level_0_top1'] > 0:
        recovery = results['level_2_top1'] / results['level_0_top1'] * 100
        results['recovery_pct'] = recovery
        print(f"\n Level-2 recovery: {recovery:.1f}% of Level-0 Top-1")
        if recovery >= 80:
            print("   STRONG RESULT (≥80% recovery)")
        elif recovery >= 75:
            print("   SUCCESS (≥75% recovery)")
        else:
            print("    Below target (<75% recovery)")

    # Model efficiency
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"\n Parameters: {total_params:.1f}M")

    # FLOPs estimation
    try:
        from fvcore.nn import FlopCountAnalysis
        privacy_cfg = PrivacyTransform.LEVEL_CONFIGS[CONFIG['privacy_level']]
        dummy = torch.randn(1, 3, CONFIG['num_frames'],
                            privacy_cfg['size'], privacy_cfg['size']).to(device)
        flops = FlopCountAnalysis(model, dummy)
        gflops = flops.total() / 1e9
        results['gflops'] = gflops
        print(f" GFLOPs: {gflops:.2f}")
    except Exception as e:
        print(f"   (FLOPs estimation skipped: {e})")

    # FPS measurement
    print("\n  Measuring inference FPS...")
    model.eval()
    privacy_cfg = PrivacyTransform.LEVEL_CONFIGS[CONFIG['privacy_level']]
    dummy = torch.randn(1, 3, CONFIG['num_frames'],
                        privacy_cfg['size'], privacy_cfg['size']).to(device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy)
    torch.cuda.synchronize()

    # Timed runs
    n_runs = 100
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy)
    torch.cuda.synchronize()
    fps = n_runs / (time.time() - t0)
    results['fps'] = fps
    print(f"  Inference FPS: {fps:.1f} clips/sec (batch=1)")

    return results


# Load best checkpoint and evaluate
best_ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'priv_vit_best.pt')
if os.path.exists(best_ckpt_path):
    best_ckpt = torch.load(best_ckpt_path, map_location=device)
    student_model.load_state_dict(best_ckpt['model'])
    print(f" Loaded best checkpoint (epoch {best_ckpt['epoch']+1})")

results = full_evaluation(student_model)


## Cell 11 — Visualization & Results

Training curves, privacy level comparison, and sample visualizations.

In [ ]:
def plot_training_curves(history):
    """Plot training loss and accuracy curves."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0].plot(epochs, history['train_loss'], 'b-', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].grid(True, alpha=0.3)

    # Train accuracy
    axes[1].plot(epochs, history['train_acc'], 'g-', linewidth=2, label='Train')
    axes[1].plot(epochs, history['val_top1'], 'r-', linewidth=2, label='Val Top-1')
    axes[1].plot(epochs, history['val_top5'], 'r--', linewidth=2, label='Val Top-5')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy Curves')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Privacy level comparison (from results)
    if results:
        levels = [0, 1, 2]
        top1 = [results.get(f'level_{l}_top1', 0) for l in levels]
        top5 = [results.get(f'level_{l}_top5', 0) for l in levels]

        x = np.arange(len(levels))
        width = 0.35
        axes[2].bar(x - width/2, top1, width, label='Top-1', color='steelblue')
        axes[2].bar(x + width/2, top5, width, label='Top-5', color='coral')
        axes[2].set_xlabel('Privacy Level')
        axes[2].set_ylabel('Accuracy (%)')
        axes[2].set_title('Accuracy vs Privacy Level')
        axes[2].set_xticks(x)
        axes[2].set_xticklabels(['Level 0\n(224px)', 'Level 1\n(112px+face)',
                                  'Level 2\n(56px+blur)'])
        axes[2].legend()
        axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    save_path = os.path.join(CONFIG['log_dir'], 'training_curves.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f" Saved to {save_path}")


def visualize_privacy_levels():
    """Show sample frames at each privacy level."""
    # Load a sample clip
    sample_dir = os.path.join(CONFIG['processed_dir'], 'val')
    sample_files = glob.glob(os.path.join(sample_dir, '*.npz'))

    if not sample_files:
        print("  No validation clips found for visualization.")
        return

    data = np.load(sample_files[0])
    frames = data['frames']  # (T, H, W, C)
    frame = frames[0]  # First frame

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    titles = ['Level 0: 224×224 (baseline)',
              'Level 1: 112×112 (face blur)',
              'Level 2: 56×56 (blur+JPEG)']

    for level in range(3):
        pt = PrivacyTransform(level=level)
        single_frame = frames[0:1]  # (1, H, W, C)
        processed = pt(single_frame)  # (C, 1, H', W')

        # Denormalize for display
        img = processed[:, 0, :, :].permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img * std + mean
        img = np.clip(img, 0, 1)

        axes[level].imshow(img)
        axes[level].set_title(titles[level], fontsize=12)
        axes[level].axis('off')

    plt.suptitle('Privacy Transform Levels', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_path = os.path.join(CONFIG['log_dir'], 'privacy_levels.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f" Saved to {save_path}")


def print_results_table(results):
    """Print a formatted results table."""
    print("\n" + "=" * 60)
    print("  FINAL RESULTS — Priv-ViT (ViViT-S + Distillation)")
    print("=" * 60)
    print(f"  {'Metric':<30} {'Value':>15}")
    print("-" * 60)

    for level in [0, 1, 2]:
        t1 = results.get(f'level_{level}_top1', 0)
        t5 = results.get(f'level_{level}_top5', 0)
        print(f"  Level {level} Top-1 Accuracy     {t1:>14.2f}%")
        print(f"  Level {level} Top-5 Accuracy     {t5:>14.2f}%")

    if 'recovery_pct' in results:
        print("-" * 60)
        print(f"  Level-2 Recovery           {results['recovery_pct']:>14.1f}%")

    if 'fps' in results:
        print(f"  Inference FPS              {results['fps']:>14.1f}")
    if 'gflops' in results:
        print(f"  GFLOPs                     {results['gflops']:>14.2f}")

    total_params = sum(p.numel() for p in student_model.parameters()) / 1e6
    print(f"  Parameters                 {total_params:>13.1f}M")
    print("=" * 60)

    # Success criterion check
    if 'recovery_pct' in results:
        rec = results['recovery_pct']
        if rec >= 80:
            print("\n STRONG POSITIVE RESULT: ≥80% recovery achieved!")
        elif rec >= 75:
            print("\n SUCCESS CRITERION MET: ≥75% recovery achieved!")
        else:
            print(f"\n  Below target: {rec:.1f}% (need ≥75%)")


# Run visualizations
print("Generating visualizations...\n")
plot_training_curves(history)
visualize_privacy_levels()
print_results_table(results)

print("\n All done! Check your Google Drive for saved checkpoints and logs.")
print(f"   {CONFIG['drive_root']}")